<a id="unit-u.218a9e3bc1f74f45af69e573894febe7"></a>

<div>
  <div>13-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Бөлімдерге қызмет көрсету және қалпына келтіру</div>
</div>

Тапсырыстар келе береді, ескі айлар кетуі, қате кері қайтарылуы керек. Үш айлық бөлімді жеке кестеде қызмет көрсету дәлелін жазып, қаңтарды `TRUNCATE` тазалайсыз, өзгерген ID-лерін көресіз; кейін тазалау алып тастаған бөлім ID-імен себеттен қайтарасыз. Себет қайтара алатын `DROP PARTITION` салыстырып, жоймас бұрын сақтау жоспарын SQL-мен тексересіз. Қосымша 6–8-бөлімдер жергілікті объектілік сақтауға көшірме алып, кестені біржола жойып, басқа базада қалпына келтіреді. Шамамен 60 минут, қосымшаға тағы 15 жоспарлаңыз.

**13-модуль: Сақтау мен өмірлік циклді басқару** · [Сабақ](course.md) · [Тест](quiz13_storage_lifecycle.ipynb) · [Курстың README файлы](../../README.md)

<a id="unit-u.ae6b7c588287453586a06d2fdd8476cf"></a>

<a id="before-you-begin"></a>

## Бастамас бұрын

Тек сынақ ортасы қажет, бұрынғы жұмыс кестелері керек емес. `ops_orders_l3`, `ops_orders_drop_l3`, жаттығуда `retention_plan_l3` қайта құрылады. Тазаланған/жойылған бөлімдер FE `catalog_trash_expire_second` баптауының 86400 секунды, бір күн, себетте қалады, кейін Doris жояды.

Қосымша 6–8-бөлімдер тек қосқанда орындалады. `doris-warehousing-backup` Compose жобасын, `doris-warehousing-backup-silo-1` контейнері мен `backup-objects` томы бар сақтауды бастап, `<your database>_backup_l3` репозиторийін және `<your database>_restore_l3` базасын құрады. Snapshot қарауға жұмыстан кейін қалады; жоюын 8-бөлім айтады.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

import pymysql

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.ui import card, show_records, show_sql

lab = connect_sandbox(module=13)
PARTITION_COLUMNS = ["PartitionId", "PartitionName", "VisibleVersion", "State", "Range", "Buckets", "ReplicationNum", "RowCount"]

<a id="unit-u.1953f0ed1e5142c889f949f5fc5341fc"></a>

<a id="1-create-a-monthly-table-of-its-own"></a>

## 1. Жеке айлық кесте құру

Бөлімдер өмірлік цикл шекарасын, HASH тарату ішкі tablet-терді анықтайды. Сынақ ортасының бір bucket/репликасы құрылымды көрсетеді, өндірістік өлшем ұсынысы емес.

Қайталау үшін бастапқы ұяшық бұрынғы кестені жояды. Орындамас бұрын DDL (Data Definition Language, дерек анықтау тілі) мен шекараларын оқыңыз.

In [ ]:
lab.execute("DROP TABLE IF EXISTS ops_orders_l3")
lab.execute("""
CREATE TABLE ops_orders_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES [("2025-01-01"), ("2025-02-01")),
    PARTITION p202502 VALUES [("2025-02-01"), ("2025-03-01")),
    PARTITION p202503 VALUES [("2025-03-01"), ("2025-04-01"))
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("ops_orders_l3", ["order_date", "order_id", "status", "amount"], [("2025-01-15", 130001, "PAID", 100), ("2025-02-15", 130002, "PAID", 200), ("2025-03-15", 130003, "SHIPPED", 300)])
lab.sql("SELECT * FROM ops_orders_l3 ORDER BY order_date, order_id", title="The lifecycle table")
expect(lab.query("""
SELECT CAST(order_date AS STRING), order_id, status, amount
FROM ops_orders_l3
ORDER BY order_date, order_id
"""), [("2025-01-15", 130001, "PAID", "100.00"), ("2025-02-15", 130002, "PAID", "200.00"), ("2025-03-15", 130003, "SHIPPED", "300.00")],
       title="One order in each of three months")

<a id="unit-u.39f1cb0eadd5499f92c401aa60c91d4f"></a>

**Күтілетін нәтиже**

Үш тапсырыс: 2025-01-15 күнгі 130001 (PAID, 100.00), 2025-02-15 күнгі 130002 (PAID, 200.00), 2025-03-15 күнгі 130003 (SHIPPED, 300.00), жасыл тексеру. Әрқайсысы өз бөлімінде: `p202501`, `p202502`, `p202503`.

<a id="unit-u.9258522be865431bb74e5e9518886f0f"></a>

<a id="2-record-the-maintenance-evidence"></a>

## 2. Қызмет көрсету дәлелін жазу

Өзгеріске дейін қамтуы мүмкінін жазып, кейін сол тексерулермен әсерін көріңіз:

- `SHOW PARTITIONS`: атау, аралық, ID, көрінетін нұсқа.
- `SHOW TABLETS`: tablet ID-і, backend, нұсқа. Әр жол — бір реплика; бөлім бағаны жоқ, бір бөлімді көруге `PARTITION (...)` қосыңыз.
- `SHOW CREATE TABLE`: кілт моделі, бөлімдер, bucket-тер, репликалар.

`RowCount` BE мерзімдік есебінен келіп, жүктеуден кейін бір минуттай -1 не 0 болуы мүмкін; `VisibleVersion` бірден өзгереді. Қазіргі дәл санаққа `SELECT COUNT(*) FROM ops_orders_l3 PARTITION (p202501)` орындаңыз. Бір bucket/реплика сынақ ортасына сай, өндірістік өлшем емес.

In [ ]:
before_partitions = lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="The partitions before maintenance",
                            columns=PARTITION_COLUMNS)
lab.sql("SHOW TABLETS FROM ops_orders_l3", title="The tablets: one row per replica",
        columns=["TabletId", "BackendId", "Version", "RowCount", "State"])
lab.sql("SHOW CREATE TABLE ops_orders_l3", title="The table's definition")
expect([name for name in before_partitions["PartitionName"]], ["p202501", "p202502", "p202503"], title="Three monthly partitions")
expect([lab.query(f"SELECT COUNT(*) FROM ops_orders_l3 PARTITION ({name})")[0][0] for name in ("p202501", "p202502", "p202503")],
       [1, 1, 1], title="One order in each partition, counted now")

<a id="unit-u.123c886d01194702be746abf21024902"></a>

**Күтілетін нәтиже**

`p202501`, `p202502`, `p202503` үш бөлімі, әрқайсысы `NORMAL`, `VisibleVersion` 2, 1 bucket/реплика; BE хабарламаса `RowCount` -1 не 0, әйтпесе 1. Бір backend-те үш tablet, әр бөлімге бір tablet, нұсқасы 2. Анықтамасы: `DUPLICATE KEY(order_date, order_id)`, үш `RANGE`, `DISTRIBUTED BY HASH(order_id) BUCKETS 1`. Екі жасыл тексеру: үш бөлім және дәл қазір әрқайсысында бір тапсырыс.

Аралықтардың төменгі/жоғарғы шегі анық: қаңтар `[2025-01-01, 2025-02-01)`, ақпан `2025-02-01`, наурыз `2025-03-01` басталады. Атау шекараны анықтамайды; мысалда қаңтарға дейінгі күндерге бөлім жоқ.

Қаңтардың `PartitionId` мәнін белгілеңіз: келесі қадам өзгертеді, 4-қадамға ескісі қажет.

<a id="unit-u.b70cc07ad8964af9a1dcaa831a8abf4f"></a>

<a id="3-clear-january-only"></a>

## 3. Тек қаңтарды тазалау

`TRUNCATE TABLE ... PARTITION (p202501)` бір бөлімді босатып, қайта жүктеуге анықтамасын сақтайды. Doris оны аты/аралығы бірдей жаңа босымен алмастырып, ескісін дерегімен себетке көшіреді. Жолдап жоймайды, compaction күтпейді: ескі жол сұраудан бірден кетеді.

Қай бөлім жаңа `PartitionId` алып, қайсысы сақтайтынын болжаңыз.

In [ ]:
lab.execute("TRUNCATE TABLE ops_orders_l3 PARTITION (p202501)")
lab.sql("SELECT * FROM ops_orders_l3 ORDER BY order_date, order_id", title="The rows after clearing January")
after_partitions = lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="The partitions after maintenance",
                           columns=PARTITION_COLUMNS)
before_ids = before_partitions.set_index("PartitionName")["PartitionId"].to_dict()
after_ids = after_partitions.set_index("PartitionName")["PartitionId"].to_dict()
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_l3"), [(2, "500.00")], title="February and March remain: 2 orders, 500.00")
expect((before_ids["p202501"] != after_ids["p202501"], [before_ids[n] == after_ids[n] for n in ("p202502", "p202503")]),
       (True, [True, True]), title="January has a new PartitionId; February and March keep theirs")
expect([lab.query(f"SELECT COUNT(*) FROM ops_orders_l3 PARTITION ({name})")[0][0] for name in ("p202501", "p202502", "p202503")],
       [0, 1, 1], title="January is empty; the other months are unchanged")

<a id="unit-u.32fcb2ae9cd64749893fe76baa6a4aa2"></a>

**Күтілетін нәтиже**

Ақпан/наурыз жолдары, кейін бөлімдер: `p202501` жаңа `PartitionId`, `VisibleVersion` 1 алады, `p202502`, `p202503` бұрынғыдай. Үш жасыл тексеру: 2 тапсырыс/500.00 қалады, ID тек қаңтарда жаңа, қаңтар бос, өзгелер өз тапсырысын сақтайды.

Нәтиже бірден өзгерді, ескі файлдар әлі ешқайда кеткен жоқ: келесі қадам қолданатын себетте.

<a id="unit-u.35570b08d60d47358388223d41c7837f"></a>

<a id="4-recover-the-truncated-january-by-its-partition-id"></a>

## 4. Тазаланған қаңтарды бөлім ID-імен қайтару

Тазалау қате болды, қаңтар қайтуы тиіс. Себетте 2-қадам жазған ID-мен ескі әлі бар. Үш кедергіні ретімен шешеді:

1. `SHOW CATALOG RECYCLE BIN WHERE NAME = 'p202501'` әр кесте/орындаудың сол аттағы барлық жойылған бөлімін тізеді; ұяшық жазылған ID-ін қалдырады.
2. Сол аттағы бос алмастырушы тұрғанда `RECOVER PARTITION p202501 FROM ops_orders_l3` өтпейді. Жоймас бұрын бос екенін тексеріңіз: кейінгі жолдар жоғалады.
3. Алмастырушыны жою оны да себетке қояды. Атаумен `RECOVER` енді ең жаңа бос `p202501` қайтарады; ұяшық ID қолданады: `RECOVER PARTITION p202501 <id> FROM ops_orders_l3`.

In [ ]:
old_id = before_ids["p202501"]
with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
    cursor.execute("SHOW CATALOG RECYCLE BIN WHERE NAME = 'p202501'")
    entries = cursor.fetchall()
show_records("The recycle bin's entry of the truncated January", [entry for entry in entries if str(entry["PartitionId"]) == str(old_id)])
try:
    lab.execute("RECOVER PARTITION p202501 FROM ops_orders_l3")
    blocked = None
except pymysql.err.OperationalError as error:
    blocked = str(error.args[-1]).split("detailMessage = ", 1)[-1]
expect(blocked is not None, True, title="RECOVER by name is refused while the replacement exists")
expect(lab.query("SELECT COUNT(*) FROM ops_orders_l3 PARTITION (p202501)"), [(0,)], title="The replacement is still empty")
recover_sql = ["ALTER TABLE ops_orders_l3 DROP PARTITION p202501", f"RECOVER PARTITION p202501 {old_id} FROM ops_orders_l3"]
show_sql("Remove the empty replacement, then recover January by its ID", ";\n".join(recover_sql) + ";")
for statement in recover_sql:
    lab.execute(statement)
recovered = lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="The partitions after the recovery",
                    columns=PARTITION_COLUMNS)
show_records("What Doris answered the RECOVER by name", [{"statement": "RECOVER PARTITION p202501 FROM ops_orders_l3", "answer": blocked}])
expect((recovered.set_index("PartitionName")["PartitionId"].to_dict()["p202501"], lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_l3")),
       (old_id, [(3, "600.00")]), title="January is back with its original ID and order: 3 orders, 600.00")

<a id="unit-u.552b341686ff4b5fb2c51b9e2afec832"></a>

**Күтілетін нәтиже**

Себет жазбасы: түрі `Partition`, аты `p202501`, база/кесте ID-лері, жазылған `PartitionId`, тазалау уақыты. Жасыл тексеру: атаумен `RECOVER` `partition[p202501] already exist in table[ops_orders_l3]` қатесімен қабылданбады; тағы жасыл — алмастырушы әлі бос. Екі командадан кейін `p202501` бастапқы `PartitionId`, `VisibleVersion` 2 мәндерімен қайтады. Жасыл: қаңтар қайтты, кестеде 3 тапсырыс/600.00.

Doris 4.1.3 өлшемі: алмастырушы жойылған соң атаумен `RECOVER` ең жаңа босын қайтарар еді, қаңтар бос қалар еді. Себет мұнда бір күндік қысқа жол; ұзақ мерзімдісі — қайта оқылған көшірме не мұрағат.

<a id="unit-u.78f1c068063445cfb4d11154305dcd49"></a>

<a id="5-drop-a-partition-and-recover-it"></a>

## 5. Бөлімді DROP арқылы жойып, RECOVER арқылы қайтару

`TRUNCATE` қайта жүктеуге сол атпен бос бөлім қалдырды. `DROP PARTITION` бөлім мен анықтамасын жояды; мұрағаты тексерілген мерзімі өткен айға осы керек.

Салыстыруға екінші жеке кестені қолданыңыз. Алдымен болжаңыз: қаңтар жойылса, SHOW PARTITIONS тек ақпан тізеді. FORCE болмаса, себет мерзімінде қайтару қаңтар мен бастапқы жолын қалпына келтіреді.

Бұл уақытша жол ұзақ мерзімді сақтық көшірмені алмастырмайды.

In [ ]:
lab.execute("DROP TABLE IF EXISTS ops_orders_drop_l3")
lab.execute("""
CREATE TABLE ops_orders_drop_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES [("2025-01-01"), ("2025-02-01")),
    PARTITION p202502 VALUES [("2025-02-01"), ("2025-03-01"))
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("ops_orders_drop_l3", ["order_date", "order_id", "amount"], [("2025-01-15", 130101, 100), ("2025-02-15", 130102, 200)])
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_drop_l3"), [(2, "300.00")], title="The second table has two months: 2 orders, 300.00")
lab.execute("ALTER TABLE ops_orders_drop_l3 DROP PARTITION p202501")
dropped_partitions = lab.sql("SHOW PARTITIONS FROM ops_orders_drop_l3 ORDER BY PartitionName", title="The partitions after the DROP", columns=PARTITION_COLUMNS);
expect("p202501" in set(dropped_partitions["PartitionName"]), False, title="The DROP removed January's definition")
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_drop_l3"), [(1, "200.00")], title="Only February's 200.00 remains")
lab.execute("RECOVER PARTITION p202501 FROM ops_orders_drop_l3")
recovered_partitions = lab.sql("SHOW PARTITIONS FROM ops_orders_drop_l3 ORDER BY PartitionName", title="The partitions after the RECOVER", columns=PARTITION_COLUMNS);
expect("p202501" in set(recovered_partitions["PartitionName"]), True, title="The RECOVER brought January's definition back")
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM ops_orders_drop_l3"), [(2, "300.00")], title="And its order: 2 orders, 300.00")

<a id="unit-u.84336f018e734c6994d52a2d09287600"></a>

**Күтілетін нәтиже**

Бес жасыл тексеру: екінші кестеде екі ай, 2 тапсырыс/300.00; `DROP` соң тек `p202502`, ақпанның 200.00 сомасы қалды; `RECOVER` соң `p202501` қайта тізіліп, 2 тапсырыс/300.00. Екі бөлім кестесі алдымен тек ақпанды, кейін екеуін көрсетеді.

Бұл `DROP` `FORCE` қоспады, бөлім себетке түсіп қайтты; мұнда атаумен `RECOVER` жеткілікті, өйткені кестенің басқа `p202501` бөлімі жоқ. `FORCE` болса не мерзім бітсе, тек сақтық көшірме/мұрағат қайтарады.

<a id="unit-u.0f9c743fd17448ad8e4b4f9ea091783f"></a>

<a id="6-optional-back-up-a-table-to-a-repository"></a>

## 6. Қосымша: кестенің сақтық көшірмесін репозиторийге алу

**Бұл және 7–8-бөлімдер қосымша, тек қосқанда орындалады**: келесі ұяшықта `RUN_BACKUP_DRILL = True` орнатып, сол және келесі ұяшықты орындаңыз. Өткізу 13-тестке не 14-модульге әсер етпейді.

Алдында Doris себеті қолданылды. Енді **нақты толық репозиторий көшірмесін** алып, жеке бастапқы кестені мәжбүрлеп жойып, snapshot-ты бөлек оқу базасына қалпына келтіріңіз. FORCE жою себетке қайтаруды әдейі алып тастайды; қалпына келу көзі — репозиторий snapshot-ы.

Бұл бөлім 4-жұмыстағы S3-үйлесімді, курсқа тиесілі Silo-ны нақты бастайды: ең көбі 512 MiB RAM, жергілікті 51910 порт (`DW_BACKUP_S3_PORT`, өзгеге сол мәнмен Jupyter бастаңыз), тұрақты `backup-objects` томы. Көмекші бастамас бұрын портты тексереді. Тек жергілікті ашық оқу тіркелгі деректерін қолданады, өндірістік тіркелгі деректерін қолданбайды. FE мен BE екеуі контейнер соңғы нүктесіне жетуі керек. Көшірілетін — Storage Policy не асинхронды көрінісі жоқ біріктірілген ортадағы кәдімгі кесте.

**Шекара:** Silo мен Doris бір хостта. Бұл хост жоғалуының апаттық қалпына келуін емес, репозиторий протоколы мен деректі салыстыруды тексереді. Өндірісте бөлек ақау домендері, тіркелгі деректерін басқару, сақтау мерзімі, сыналған қалпына келу мақсаттары қажет.

In [ ]:
# Optional: set to True, then run this cell and the next one, to back up and restore a table.
RUN_BACKUP_DRILL = False

if RUN_BACKUP_DRILL:
    import secrets
    from dw_course.backup import prepare_backup_storage, BUCKET, ACCESS_KEY, SECRET_KEY
    from dw_course.streaming import wait_for

    backup_client = prepare_backup_storage(lab, start=True)
    repository = lab.database + "_backup_l3"
    restore_database = lab.database + "_restore_l3"
    backup_label = "course_snapshot_" + secrets.token_hex(6)
    existing_repositories = {row[1] for row in lab.query("SHOW REPOSITORIES")}
    if repository in existing_repositories:
        lab.execute(f"DROP REPOSITORY {repository}")
    lab.execute(f"""CREATE REPOSITORY {repository} WITH S3
    ON LOCATION "s3://{BUCKET}/{lab.database}/repository"
    PROPERTIES ("s3.endpoint"="http://course-backup-silo:9000", "s3.region"="us-east-1",
                "s3.access_key"="{ACCESS_KEY}", "s3.secret_key"="{SECRET_KEY}", "use_path_style"="true")""")
    lab.execute("DROP TABLE IF EXISTS backup_source_l3")
    lab.execute("""CREATE TABLE backup_source_l3 (
        order_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
    ) DUPLICATE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
    PROPERTIES ("replication_num"="1")""")
    backup_rows = [(7301,"100.00"),(7302,"200.00"),(7303,"300.00")]
    lab.insert("backup_source_l3", ["order_id","amount"], backup_rows)
    expect(lab.query("SELECT * FROM backup_source_l3 ORDER BY order_id"), backup_rows, title="The source table holds the three rows to back up")
    lab.execute(f'BACKUP SNAPSHOT {lab.database}.{backup_label} TO {repository} ON (backup_source_l3) PROPERTIES("type"="full")')

    def repository_job(kind, database):
        with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
            cursor.execute(f"SHOW {kind} FROM {database}")
            name_column = "SnapshotName" if kind=="BACKUP" else "Label"
            jobs = [row for row in cursor.fetchall() if row[name_column] == backup_label]
        if not jobs:
            return None
        current = jobs[-1]
        if current["State"] == "CANCELLED":
            raise RuntimeError(f"{kind} failed: {current}")
        return current

    backup_job = wait_for(lambda: repository_job("BACKUP",lab.database),
                          lambda row: row is not None and row["State"]=="FINISHED",
                          description="Upload the full snapshot to the repository", timeout=180)
    lab.sql(f"SHOW BACKUP FROM {lab.database}", title="The backup jobs of your database; this run's is the last");
else:
    card("Set RUN_BACKUP_DRILL = True in this cell and run it again to start the object store and back up a table.",
         "skip", "The optional backup drill was not run")

<a id="unit-u.af05ec733f5e469bbc888f815e5b606d"></a>

**Күтілетін нәтиже**

`RUN_BACKUP_DRILL = False` болса сұр карточка, қосымша орындалмады.

`True` болса: сақтауды бастаудың үш жасыл қадамы; 7301–7303 үш бастапқы жолдың жасыл тексеруі; snapshot жүктелгенде барыс карточкасы; база көшірме тапсырмалары — осы орындалудың соңғысы `course_snapshot_<random>` snapshot атауымен `FINISHED`, бұрынғылары үстінде.

<a id="unit-u.b13d588acc86400a92f34d21d747ef38"></a>

<a id="7-optional-restore-into-another-database-and-reconcile-every-row"></a>

## 7. Қосымша: басқа базаға қалпына келтіріп, әр жолды салыстыру

Сәтті жүктеу — сыналған қалпына келу емес. Уақытты ойдан шығармай, нақты snapshot timestamp алыңыз; курс префиксіне сай бөлек базаға қайтарыңыз. `ON (... AS ...)` жалғыз қайтарылатын кестені атайды; басқалары мен негізгі база тазаланбайды.

Бастапқы кесте тек көшірме `FINISHED` болғаннан кейін FORCE жойылады. Аяқталуын, реттелген жолдарды, сан/соманы тексеріңіз. Snapshot өз шекарасын қайтарады, кейінгі жазуларды емес; дерек жоғалтудың рұқсат етілген аралығын азайту үшін басқа қайта ойнату тәсілі керек.

In [ ]:
if RUN_BACKUP_DRILL:
    snapshot = lab.sql(f"SHOW SNAPSHOT ON {repository} WHERE SNAPSHOT='{backup_label}'", title="Repository snapshot and real backup timestamp")
    expect(snapshot.iloc[0]["Status"], "OK")
    backup_timestamp = snapshot.iloc[0]["Timestamp"]
    lab.execute("DROP TABLE backup_source_l3 FORCE")
    lab.execute(f"CREATE DATABASE IF NOT EXISTS {restore_database}")
    lab.execute(f"DROP TABLE IF EXISTS {restore_database}.restored_orders_l3")
    lab.execute(f"""RESTORE SNAPSHOT {restore_database}.{backup_label} FROM {repository}
    ON (backup_source_l3 AS restored_orders_l3)
    PROPERTIES ("backup_timestamp"="{backup_timestamp}", "replication_num"="1")""")
    restore_job = wait_for(lambda: repository_job("RESTORE",restore_database),
                           lambda row: row is not None and row["State"]=="FINISHED",
                           description="Restore the repository snapshot into an isolated target", timeout=180)
    expect(lab.query(f"SELECT * FROM {restore_database}.restored_orders_l3 ORDER BY order_id"), backup_rows, title="Restored rows match the independent backup-cutoff ledger")
    expect(lab.query(f"SELECT COUNT(*),SUM(amount) FROM {restore_database}.restored_orders_l3"), [(3,"600.00")], title="The restored table has 3 orders and 600.00")
    show_records("Recovery source comparison", [
        {"operation":"Earlier RECOVER drill","source":"recycle bin","boundary":"retention and original metadata must still exist"},
        {"operation":"This RESTORE drill","source":"full repository snapshot","boundary":"snapshot cutoff; no post-snapshot writes"},
    ])
    lab.sql(f"SHOW RESTORE FROM {restore_database}", title="The restore jobs; this run's is the last");
else:
    card("Run the previous cell with RUN_BACKUP_DRILL = True first.", "skip", "The optional restore was not run")

<a id="unit-u.da7963d35c09416e9efb22023d0ef7a0"></a>

**Күтілетін нәтиже**

Өшірулі болса сұр карточка.

Қосулы болса: репозиторий snapshot-ы `Status` OK және нақты уақыт белгісімен; қайтаруда барыс карточкасы; екі жасыл тексеру — 7301–7303 үш жолы, 3 тапсырыс/600.00; себет пен репозиторий жолдарын салыстыру; қалпына келтіру тапсырмалары, осы орындаудың соңғысы `FINISHED`.

Бастапқы `FORCE` жойылған, себет көмектеспеді, snapshot көмектесті. Ол көшірме алынған кездегіні, кейінгі жазуларды емес, сақтайды.

<a id="unit-u.903f3bdbe93a4da69d8d7275dedc101d"></a>

<a id="8-optional-keep-the-snapshot-and-own-its-cleanup"></a>

## 8. Қосымша: snapshot-ты сақтап, тазалауына жауап беру

Репозиторий, Silo snapshot нысандары, бөлек қайтарылған кесте тексеруге қалады. Қайталау жаңа snapshot белгісін жасап, тек аталған бастапқы/нысаналы кестелерді қайта құрады. Сақтау томын ешқашан жоймайды. RAM босатып, нысандарды сақтау үшін курс түбірінен `docker compose --project-name doris-warehousing-backup --file environments/backup/compose.yml stop` орындаңыз.

Repository жою — Doris тіркеуін жою, қашық snapshot дерегін емес. Көшірмені тек нақты сақтау шешімімен жойыңыз; сервисті тоқтату жою емес. Бүкіл кластер, рұқсаттар, сыртқы lake дерегі көшірілмейді. Бәрін жоюға Doris ішінде `DROP REPOSITORY <your database>_backup_l3`, `DROP DATABASE <your database>_restore_l3`, кейін курс каталогынан `docker compose --project-name doris-warehousing-backup --file environments/backup/compose.yml down --volumes` орындаңыз; реті README тазалау бөлімінде.

<a id="unit-u.11041263ea2d467da89563eb3c7e2307"></a>

<a id="9-practice-check-a-retention-plan-before-it-removes-anything"></a>

## 9. Жаттығу: сақтау жоспарын жоймас бұрын тексеру

Кесте осы ай мен алдыңғы 12 күнтізбелік айды сақтайды. Қызмет көрсету 2026-02-15: 2025 ақпан–2026 ақпан қалады, бұрынғылары кетеді. 2024 қараша–2026 ақпан әр айына бір бөлім/тапсырысты `retention_plan_l3` құрыңыз; қызмет көрсету күнінен SQL-мен алғашқы сақталатын айды есептеңіз; соған дейін не сол кезде бітетіндерді тізіп, `FORCE` қоспай жойыңыз; 2025 ақпан–2026 ақпан 13 ай/13 тапсырыс қалғанын тексеріңіз.

In [ ]:
# Your code here

<a id="unit-u.439a67162c17471b9176aa0fb05377ab"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
months = [f"{2024 + (10 + i) // 12}-{(10 + i) % 12 + 1:02d}" for i in range(16)]  # 2024-11 ... 2026-02
bounds = months[1:] + ["2026-03"]
partitions = ",\n    ".join(f'PARTITION p{m.replace("-", "")} VALUES [("{m}-01"), ("{b}-01"))' for m, b in zip(months, bounds))
plan_ddl = f"""CREATE TABLE retention_plan_l3 (
    order_date DATE NOT NULL, order_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    {partitions}
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES ("replication_num"="1")"""
lab.execute("DROP TABLE IF EXISTS retention_plan_l3")
show_sql("A table of 16 monthly partitions", plan_ddl)
lab.execute(plan_ddl)
lab.insert("retention_plan_l3", ["order_date", "order_id", "amount"], [(f"{m}-15", 150000 + i, 10) for i, m in enumerate(months)])
keep_from = lab.query("SELECT CAST(DATE_TRUNC(DATE_SUB(DATE '2026-02-15', INTERVAL 12 MONTH), 'month') AS STRING)")[0][0]
expired = [f"p{m.replace('-', '')}" for m, b in zip(months, bounds) if f"{b}-01" <= keep_from]
show_records("The plan", [{"maintenance_date": "2026-02-15", "first_month_kept": keep_from, "partitions_to_drop": ", ".join(expired)}])
for name in expired:
    lab.execute(f"ALTER TABLE retention_plan_l3 DROP PARTITION {name}")
expect((keep_from, expired), ("2025-02-01", ["p202411", "p202412", "p202501"]), title="The plan drops November 2024 to January 2025")
expect(lab.query("SELECT COUNT(*), CAST(MIN(order_date) AS STRING), CAST(MAX(order_date) AS STRING) FROM retention_plan_l3"),
       [(13, "2025-02-15", "2026-02-15")], title="13 months remain, February 2025 to February 2026")

<a id="unit-u.ffed003466714b3a8e8eee022e943abf"></a>

**Күтілетін нәтиже**

`p202411`–`p202602` 16 бөлім анықтамасы, кейін жоспар: күн 2026-02-15, алғашқы сақталатын ай 2025-02-01, жойылатындар `p202411, p202412, p202501`. Екі жасыл тексеру: үшеуі жойылады, 2025-02-15–2026-02-15 аралығында 13 тапсырыс қалады.

Ереже күнтізбелік аймен; 365 күндік жылжымалы терезе басқа күнде бітеді, тұтас ай жоюмен орындалмайды. Нақты іске қосуға 2-қадам тексеруі мен әр ай мұрағатын қайта оқуды қосыңыз; `FORCE` болмаса жойылған айлар бір күн себетте.

<a id="unit-u.3534631d423b4749a027bc92ac5aa333"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- `SHOW PARTITIONS`, `SHOW TABLETS`, `SHOW CREATE TABLE`, бөлім `COUNT(*)` әрқайсысы не көрсететінін, қай сан кешігетінін айта аласыз;
- `TRUNCATE` тек қаңтарды босатып, жаңа `PartitionId` берді; файл жойылмай тұрып нәтиже неге өзгергенін білесіз;
- қаңтарды ID-мен қайтарып, атаумен `RECOVER` алдымен неге өтпей, кейін қате бөлімді қайтаратынын білесіз;
- қайта жүктеуге `TRUNCATE`, мерзімі өткенге `DROP PARTITION` таңдап, себет нені алмастыратынын/алмастырмайтынын айта аласыз;
- жоспар 2024 қараша–2025 қаңтарды жойып, 13 ай сақтады.

<a id="unit-u.900656a959a74dd59697215e5a79d4ec"></a>

<a id="next"></a>

## Әрі қарай

[13-тест: Сақтау мен өмірлік циклді басқару](quiz13_storage_lifecycle.ipynb) файлын ашып, [14-модуль: Ресурстарды оқшаулау және сұрауларды қорғау](../module14-resource-isolation/course.md) бөліміне өтіңіз.